# Pandas — Handling NaN

Pandas' aggregation methods (`mean`, `sum`, `max`, `count`, etc.) **automatically skip NaN values** by default — this is the most important thing to know about missing data.

```
df['col'].mean()   → NaN is SKIPPED (skipna=True default)
                   → mean = sum_of_non_NaN / count_of_non_NaN
```

This is the **opposite** of NumPy, which poisons aggregations when NaN is present.

In [2]:
import pandas as pd
import numpy as np

## Setup — DataFrame with NaN

Create a DataFrame where one row has missing values (NaN salary, None city).

In [3]:
df = pd.DataFrame({
    'name':   ['Alice', 'Bob',  'Carol', 'Dave',  'Carol'],
    'age':    [25,      30,     35,      40,      35],
    'salary': [50000,   60000,  75000,   90000,   np.nan],   # last row NaN
    'city':   ['NYC',   'LA',   'NYC',   'SF',    None]      # last row None
}, index=['e1', 'e2', 'e3', 'e4', 'e5'])
df

,name,age,salary,city
e1,Alice,25,50000.0,NYC
e2,Bob,30,60000.0,LA
e3,Carol,35,75000.0,NYC
e4,Dave,40,90000.0,SF
e5,Carol,35,NaN,None


In [8]:
df.value_counts() # useless gives 1

name   age  salary   city
Alice  25   50000.0  NYC     1
Bob    30   60000.0  LA      1
Carol  35   75000.0  NYC     1
Dave   40   90000.0  SF      1
Name: count, dtype: int64

In [10]:
df['name'].value_counts()

name
Carol    2
Alice    1
Bob      1
Dave     1
Name: count, dtype: int64

In [14]:

print(df['name'].nunique())
df['name'].unique()

4


array(['Alice', 'Bob', 'Carol', 'Dave'], dtype=object)

## Mean Skips NaN by Default

```
Sum of non-NaN values = 50000 + 60000 + 75000 + 90000 = 275000
Count of non-NaN values = 4    (NOT 5!)

Mean = 275000 / 4 = 68,750
```

NaN is **excluded entirely** — not treated as 0, not counted toward the divisor.

In [4]:
df['salary'].mean()    # → 68750.0  (NaN skipped)

np.float64(68750.0)

## skipna=False — Force NaN to Poison

If you want NaN to propagate (single NaN → entire result is NaN), pass `skipna=False`:

In [5]:
df['salary'].mean(skipna=False)   # → NaN

np.float64(nan)

## Other Aggregations Behave Similarly

All these default to `skipna=True`:

In [ ]:
print('count:  ', df['salary'].count())       # 4  (non-NaN only)
print('sum:    ', df['salary'].sum())          # 275000.0
print('mean:   ', df['salary'].mean())         # 68750.0
print('max:    ', df['salary'].max())          # 90000.0
print('min:    ', df['salary'].min())          # 50000.0
print('std:    ', df['salary'].std())          # std of 4 values
print('size:   ', df['salary'].size)           # 5 (INCLUDING NaN — total length)
print('isna sum:', df['salary'].isna().sum())  # 1 (count of NaNs)

Note the difference: `count()` gives **non-NaN** count, while `size` gives the **total length** (including NaN).

## Pandas vs NumPy — Opposite Defaults

Pandas skips NaN. NumPy poisons. Same operation, different behaviour.

In [ ]:
arr = np.array([50000, 60000, 75000, 90000, np.nan])

print('numpy mean():        ', np.mean(arr))     # NaN — poisoned
print('numpy nanmean():     ', np.nanmean(arr))  # 68750.0 — skips NaN
print('pandas Series.mean():', pd.Series(arr).mean())  # 68750.0 — skips by default

## Treating NaN as Zero — fillna(0)

If you want NaN to be counted as 0 (and included in the count):

In [15]:
df['salary'].fillna(0).mean()
# (50000 + 60000 + 75000 + 90000 + 0) / 5 = 55000.0

np.float64(55000.0)

In [ ]:
df['salary'].fillna(0).mean()

Different result because the NaN is now treated as 0 AND counted toward the divisor.

## NaN in groupby

Group means also skip NaN per group:

In [ ]:
df.groupby('city')['salary'].mean()
# city
# LA     60000.0    (just Bob)
# NYC    62500.0    (Alice + Carol)
# SF     90000.0    (just Dave)
# None group EXCLUDED by default — see next cell

By default, groupby **drops rows where the grouping key is NaN/None**. Pass `dropna=False` to keep them:

In [ ]:
df.groupby('city', dropna=False)['salary'].mean()
# city
# LA     60000.0
# NYC    62500.0
# SF     90000.0
# NaN    NaN       ← e5's group (its city is None, its salary is NaN)

## Detecting Missing Values

Common patterns for finding and counting NaNs:

In [16]:
df.isna()                # boolean DataFrame — True where NaN/None

,name,age,salary,city
e1,False,False,False,False
e2,False,False,False,False
e3,False,False,False,False
e4,False,False,False,False
e5,False,False,True,True


In [17]:
df.isna().sum()          # count of NaNs PER COLUMN

name      0
age       0
salary    1
city      1
dtype: int64

In [18]:
df.isna().sum().sum()    # total NaN count across whole DataFrame

np.int64(2)

In [22]:
df.isna().any(axis=0) # Colums of rows with NaN

name      False
age       False
salary     True
city       True
dtype: bool

In [24]:
df.isna().any(axis=1) # series of Rows with NaN


e1    False
e2    False
e3    False
e4    False
e5     True
dtype: bool

-❗ Key rule (very important)
- df[...] behaves like:

|Input inside []	|Meaning|
|--|--|
|column name ('A')|	column selection|
|list of columns (['A','B'])|	column selection|
|boolean Series (len = rows)|	row filtering|
|slice (:)|	row slicing (label-based index)|

In [19]:
# Which rows have ANY missing value?
df[df.isna().any(axis=1)] # row filtering
# df[df.isna().any(axis=0)] # row filtering - gives c1: False |c2: False => not row filter => 🚩error
# Think of df[...] as:

# “Let pandas interpret what you gave me”

# string/list → columns
# boolean array → rows

,name,age,salary,city
e5,Carol,35,NaN,None


In [21]:
# Which rows are COMPLETE (no NaN)?
df[df.notna().all(axis=1)]

,name,age,salary,city
e1,Alice,25,50000.0,NYC
e2,Bob,30,60000.0,LA
e3,Carol,35,75000.0,NYC
e4,Dave,40,90000.0,SF


## Handling Missing Values

Three main strategies:

In [ ]:
# 1. DROP rows with any NaN
df.dropna()              # default: drop rows with ANY NaN
# df.dropna(how='all')   # drop rows where ALL values are NaN
# df.dropna(subset=['salary'])  # drop only when salary is NaN

In [ ]:
# 2. FILL NaN with a value
df.fillna(0)                                  # fill ALL NaNs with 0
# df.fillna({'salary': 0, 'city': 'Unknown'})  # different value per column
# df['salary'].fillna(df['salary'].mean())     # fill with column mean

In [ ]:
# 3. INTERPOLATE — fill by linear interpolation between neighbours
df['salary'].interpolate()

## NaN vs None vs np.nan

Three things look like missing values but behave differently:

| | Type | Where pandas treats as missing |
|-|------|-------------------------------|
| `np.nan` | float | Yes |
| `None` | Python None | Yes (converted to NaN in numeric columns) |
| `pd.NA` | pandas NA scalar | Yes (modern, type-stable) |
| `''` (empty string) | str | NOT missing — it's a valid string |
| `0` | int/float | NOT missing — it's a value |

`isna()` catches NaN, None, and pd.NA. It does NOT catch empty strings or zeros.

In [ ]:
s = pd.Series([1, 2, np.nan, None, 0, ''])
print(s.isna())
# 0    False
# 1    False
# 2    True       ← np.nan
# 3    True       ← None (converted to NaN)
# 4    False      ← 0 is a valid value
# 5    False      ← empty string is a valid value

## Common Gotchas

```
❌ NaN == NaN              → False (always!)
✓ pd.isna(value)            → reliable NaN check
✓ value != value            → True only for NaN (hacky but works)

❌ Comparing NaN to anything always gives False
   df['col'] == np.nan  → returns all False
✓ df['col'].isna()      → correct way to find NaN rows

❌ Mixing int and NaN → column becomes float64
   pandas can't have NaN in int columns (use 'Int64' nullable type)
```

In [ ]:
# NaN comparison is ALWAYS False
print(np.nan == np.nan)          # False
print(np.nan != np.nan)          # True
print(pd.isna(np.nan))           # True   ← use this

## Summary

```
Pandas aggregations SKIP NaN by default:
   df['col'].mean()                → 68750.0  (NaN excluded)
   df['col'].mean(skipna=False)    → NaN     (poison)

NumPy POISONS by default:
   np.mean(arr_with_nan)           → NaN
   np.nanmean(arr_with_nan)        → mean (skips NaN)

Detection:
   df.isna()                        → boolean mask
   df.isna().sum()                  → count per column

Handling:
   df.dropna()                       → drop rows with NaN
   df.fillna(value)                  → fill with constant
   df['col'].fillna(df['col'].mean())  → fill with column mean
   df.interpolate()                  → linear interpolation

Comparison trap:
   np.nan == np.nan → False (always!)
   Use .isna() or pd.isna() to test for NaN
```

> **Pandas defaults to forgiveness with NaN.** Aware of it → it works for you. Unaware → silent bugs when your counts/means are off.